# Seascape Toolkit Validation

## 0. Purpose

This notebook is a small, offline toolkit smoke/acceptance workflow:

```text
install/import → configuration → workspace initialization → data/product discovery
               → representative execution → output inspection → visual validation → summary
```

It complements pytest. It is not an OrcaCast model, regional scientific validation, or proof that live external providers are available. The required path uses deterministic synthetic inputs and the same Seascape production API used by consumers.

To inspect a real completed Seascape release, use [`notebooks/01_DATA_EXPLORER.ipynb`](../01_DATA_EXPLORER.ipynb).

## 1. Environment and installation check

Run this notebook from the repository root after `python -m pip install -e '.[test,notebook]'`.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import time
from importlib.metadata import version
from pathlib import Path

import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seascape
import yaml
from IPython.display import display
from rasterio.transform import from_bounds
from shapely.geometry import Polygon

from seascape.products import list_products, list_resolutions, resolve_product

STARTED_AT = time.perf_counter()
CHECKS: dict[str, bool] = {}
NOTEBOOK_WORKING_DIRECTORY = Path.cwd().resolve()
REPOSITORY_ROOT = next((
    candidate for candidate in (NOTEBOOK_WORKING_DIRECTORY, *NOTEBOOK_WORKING_DIRECTORY.parents)
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'src/seascape').is_dir()
), None)
if REPOSITORY_ROOT is None:
    raise RuntimeError('Could not locate the toolkit-seascape repository root.')

PACKAGE_PATH = Path(seascape.__file__).resolve()
PACKAGE_VERSION = version('toolkit-seascape')
USING_CHECKOUT = PACKAGE_PATH.is_relative_to((REPOSITORY_ROOT / 'src').resolve())
CHECKS['Package import'] = all((seascape, list_products, list_resolutions, resolve_product))

environment = pd.Series({
    'python': sys.version.split()[0],
    'operating_system': platform.platform(),
    'toolkit_seascape_version': PACKAGE_VERSION,
    'seascape_import_path': str(PACKAGE_PATH),
    'notebook_working_directory': str(NOTEBOOK_WORKING_DIRECTORY),
    'execution_source': 'repository checkout' if USING_CHECKOUT else 'installed distribution',
    'package_import': 'PASS' if CHECKS['Package import'] else 'FAIL',
}, name='environment')
display(environment.to_frame('value'))

## 2. Initialize a clean validation workspace

The installed `seascape` CLI initializes a temporary workspace. The directory is automatically removed when this kernel exits.

In [ ]:
VALIDATION_TEMP = tempfile.TemporaryDirectory(prefix='seascape-toolkit-validation-')
WORKSPACE = Path(VALIDATION_TEMP.name).resolve()
SEASCAPE_CLI = shutil.which('seascape')
if SEASCAPE_CLI is None:
    raise RuntimeError("The 'seascape' command is unavailable; install the toolkit in this environment.")

init_run = subprocess.run(
    [SEASCAPE_CLI, '--workspace', str(WORKSPACE), 'init'],
    check=True, capture_output=True, text=True,
)
os.environ['SEASCAPE_WORKSPACE'] = str(WORKSPACE)
EXPECTED_CONFIG = [
    Path('config/common.yaml'),
    Path('config/data/project.yaml'),
    Path('config/data/environment_seascape.yaml'),
    Path('config/data/presentation_settings.yaml'),
]
CHECKS['Workspace initialized'] = all((WORKSPACE / path).is_file() for path in EXPECTED_CONFIG)
workspace_tree = sorted(
    str(path.relative_to(WORKSPACE)) + ('/' if path.is_dir() else '')
    for path in WORKSPACE.rglob('*')
    if len(path.relative_to(WORKSPACE).parts) <= 3
)
print(f'Workspace: {WORKSPACE}')
print(f'CLI: {SEASCAPE_CLI}')
print('\n'.join(workspace_tree[:80]))
assert CHECKS['Workspace initialized'], 'Workspace initialization did not create required config.'

## 3. Inspect configuration

Only the fields relevant to this validation are displayed; the full regional YAML remains the source of truth.

In [ ]:
COMMON_PATH = WORKSPACE / 'config/common.yaml'
PROJECT_PATH = WORKSPACE / 'config/data/project.yaml'
SEASCAPE_CONFIG_PATH = WORKSPACE / 'config/data/environment_seascape.yaml'
common_config = yaml.safe_load(COMMON_PATH.read_text(encoding='utf-8'))
project_config = yaml.safe_load(PROJECT_PATH.read_text(encoding='utf-8'))
seascape_config = yaml.safe_load(SEASCAPE_CONFIG_PATH.read_text(encoding='utf-8'))
bathymetry_config = seascape_config['bathymetry']
network_config = seascape_config['water_network']
model_area = common_config['areas'][bathymetry_config['area']]['bbox_wgs84']
configuration_summary = pd.Series({
    'area': bathymetry_config['area'],
    'bbox_wgs84': model_area,
    'water_network_resolutions': network_config['resolutions'],
    'bathymetry_h3_resolution': bathymetry_config['processing']['h3_resolution'],
    'bathymetry_provider': bathymetry_config['source']['provider'],
    'bathymetry_release': bathymetry_config['source']['release'],
    'bathymetry_sign': bathymetry_config['processing']['bathymetry_sign'],
    'candidate_root': str(WORKSPACE / '.seascape/candidates/seascape'),
    'configured_output': bathymetry_config['processing']['processed_path'],
    'project_include': project_config['SEASCAPE_LAYER'],
}, name='configuration')
CHECKS['Configuration loaded'] = bool(model_area and bathymetry_config and network_config)
display(configuration_summary.to_frame('value'))
assert CHECKS['Configuration loaded']

## 4. Inspect the Seascape data/product catalog

This table is derived directly from the installed dataset registry; it is not a notebook-maintained product list. The immutable `seascape.products` resolver is imported above but requires a completed canonical release, so it is exercised by the Data Explorer rather than fabricated here.

In [ ]:
from seascape.core.data import register_builtin_datasets
from seascape.core.data.registry import DATASETS

register_builtin_datasets()
catalog_rows = []
for spec in DATASETS:
    resolution_match = re.search(r'_r(\d+)(?:_|$)', str(spec.dataset_id))
    catalog_rows.append({
        'dataset_product': str(spec.dataset_id),
        'family': spec.producer.split('.')[2] if len(spec.producer.split('.')) > 2 else spec.producer,
        'dependencies': ', '.join(map(str, spec.dependencies)) or '—',
        'resolution': int(resolution_match.group(1)) if resolution_match else pd.NA,
        'grain': ' + '.join(spec.primary_key) or 'artifact',
        'output_type': spec.format.value,
    })
catalog = pd.DataFrame(catalog_rows)
CHECKS['Dataset registry available'] = len(catalog) > 0 and catalog['dataset_product'].is_unique
print(f'{len(catalog)} registered products')
display(catalog)
assert CHECKS['Dataset registry available']

## 5. Inspect the build graph

The public workflow planner resolves the same stages used by `seascape build`. A CLI dry run checks command dispatch; the compact table keeps the graph readable.

In [ ]:
from seascape.workflow import selected_stages

planned_stages = selected_stages()
build_plan = pd.DataFrame([{
    'stage': stage.name,
    'dependencies': ', '.join(stage.dependencies) or '—',
    'declared_outputs': '; '.join(stage.declared_outputs) or '—',
} for stage in planned_stages])
dry_run = subprocess.run(
    [SEASCAPE_CLI, '--workspace', str(WORKSPACE), 'build', '--dry-run'],
    check=True, capture_output=True, text=True,
)
CHECKS['Build graph resolved'] = bool(planned_stages) and dry_run.returncode == 0
display(build_plan)
print(f'CLI dry run resolved {len(planned_stages)} stages.')
assert CHECKS['Build graph resolved']

## 6. Create a tiny deterministic validation input

The fixture is generated at runtime in the temporary workspace: a 48 × 48 single-band EPSG:4326 GeoTIFF with negative-elevation meters, explicit `-9999` nodata, a smooth depth gradient, and a small missing corner. H3 r8 support and bounded neighborhoods are deterministic fixture inputs for the production bathymetry pipeline. No credentials, network calls, regional files, or tracked binary assets are used.

In [ ]:
INPUT_DIR = WORKSPACE / 'validation/input'
OUTPUT_DIR = WORKSPACE / 'validation/output'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RASTER_PATH = INPUT_DIR / 'synthetic_bathymetry.tif'
GRID_PATH = INPUT_DIR / 'H3_SUPPORT_RES_8.parquet'
NEIGHBORHOOD_PATH = INPUT_DIR / 'H3_WATER_NEIGHBORHOODS_RES_8.parquet'
OUTPUT_PATH = OUTPUT_DIR / 'BATHYMETRY.parquet'

height = width = 48
bounds = (-123.20, 48.40, -123.10, 48.50)
transform = from_bounds(*bounds, width, height)
row_grid, column_grid = np.indices((height, width), dtype='float32')
elevation = -(5.0 + 145.0 * column_grid / (width - 1) + 80.0 * row_grid / (height - 1))
elevation[-4:, -4:] = -9999.0
with rasterio.open(
    RASTER_PATH, 'w', driver='GTiff', height=height, width=width, count=1,
    dtype='float32', crs='EPSG:4326', transform=transform, nodata=-9999.0,
) as target:
    target.write(elevation.astype('float32'), 1)

longitudes = transform.c + (column_grid + 0.5) * transform.a
latitudes = transform.f + (row_grid + 0.5) * transform.e
valid = elevation != -9999.0
cells = sorted({
    h3.latlng_to_cell(float(lat), float(lon), 8)
    for lat, lon in zip(latitudes[valid], longitudes[valid], strict=True)
})
pd.DataFrame({'H3_INDEX': cells}).to_parquet(GRID_PATH, index=False)

neighborhood_rows = []
edge_length_m = float(h3.average_hexagon_edge_length(8, unit='m'))
for source in cells:
    for target in cells:
        try:
            hops = int(h3.grid_distance(source, target))
        except h3.H3BaseException:
            continue
        if hops <= 2:
            neighborhood_rows.append({
                'SOURCE_H3_INDEX': source,
                'TARGET_H3_INDEX': target,
                'MINIMUM_HOP_COUNT': hops,
                'NETWORK_DISTANCE_M': hops * edge_length_m,
            })
pd.DataFrame(neighborhood_rows).to_parquet(NEIGHBORHOOD_PATH, index=False)

validation_config = yaml.safe_load(SEASCAPE_CONFIG_PATH.read_text(encoding='utf-8'))
validation_config['base_directory'] = str(WORKSPACE)
validation_config['bathymetry']['source']['raw_dir'] = str(INPUT_DIR)
validation_config['bathymetry']['source']['raw_filename'] = RASTER_PATH.name
validation_config['bathymetry']['processing']['h3_grid_path_template'] = str(INPUT_DIR / 'H3_SUPPORT_RES_{res}.parquet')
validation_config['bathymetry']['processing']['processed_path'] = str(OUTPUT_PATH)
validation_config['bathymetry']['processing']['additional_exports'] = []
validation_config['water_network']['output_dir'] = str(INPUT_DIR)
validation_config['water_network']['neighborhood_filename_template'] = NEIGHBORHOOD_PATH.name.replace('8', '{res}')
VALIDATION_CONFIG_PATH = WORKSPACE / 'config/data/validation_seascape.yaml'
VALIDATION_CONFIG_PATH.write_text(yaml.safe_dump(validation_config, sort_keys=False), encoding='utf-8')

CHECKS['Validation input readable'] = RASTER_PATH.is_file() and GRID_PATH.is_file() and NEIGHBORHOOD_PATH.is_file()
print(f'Raster: {RASTER_PATH} ({width} × {height})')
print(f'H3 support: {len(cells)} cells')
print(f'Neighborhood rows: {len(neighborhood_rows)}')
assert CHECKS['Validation input readable']

## 7. Run one real toolkit transformation

`run_pipeline` is the public production bathymetry orchestration API. `skip_download=True` uses the configured local fixture, and `skip_map=True` avoids a second renderer while retaining transactional Parquet publication and a provenance manifest.

In [ ]:
from seascape.seafloor_physiography.bathymetry import run_pipeline

pipeline_started = time.perf_counter()
raw_path, processed_path, map_path = run_pipeline(
    config_path=VALIDATION_CONFIG_PATH,
    skip_download=True,
    skip_map=True,
)
PIPELINE_SECONDS = time.perf_counter() - pipeline_started
MANIFEST_PATH = processed_path.parent / 'bathymetry_manifest.json'
CHECKS['Production transformation completed'] = (
    raw_path == RASTER_PATH and processed_path == OUTPUT_PATH and map_path is None
)
print(f'INPUT  {raw_path}')
print('  ↓ seascape.seafloor_physiography.bathymetry.run_pipeline')
print(f'OUTPUT {processed_path}')
print(f'Elapsed: {PIPELINE_SECONDS:.2f} seconds')
assert CHECKS['Production transformation completed']

## 8. Inspect the input data

In [ ]:
with rasterio.open(RASTER_PATH) as source:
    input_values = source.read(1, masked=True)
    INPUT_CRS = source.crs
    INPUT_BOUNDS = source.bounds
    input_summary = pd.Series({
        'width': source.width, 'height': source.height, 'count': source.count,
        'crs': str(source.crs), 'transform': str(source.transform),
        'bounds': tuple(source.bounds), 'nodata': source.nodata,
        'minimum_elevation_m': float(input_values.min()),
        'maximum_elevation_m': float(input_values.max()),
        'mean_elevation_m': float(input_values.mean()),
        'valid_pixel_count': int(input_values.count()),
        'nodata_pixel_count': int(input_values.size - input_values.count()),
    }, name='input raster')
display(input_summary.to_frame('value'))

## 9. Inspect generated outputs

The Parquet grain is one unique H3 r8 cell. Bathymetry is positive-down meters because the configured source uses negative elevation and the production pipeline applies the declared sign convention.

In [ ]:
output = pd.read_parquet(OUTPUT_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
schema = pd.DataFrame({
    'column': output.columns,
    'dtype': [str(output[column].dtype) for column in output.columns],
    'null_count': [int(output[column].isna().sum()) for column in output.columns],
    'null_fraction': [float(output[column].isna().mean()) for column in output.columns],
})
output_summary = pd.Series({
    'path': str(OUTPUT_PATH), 'output_type': 'Parquet',
    'rows': len(output), 'columns': len(output.columns),
    'grain': 'unique H3_INDEX', 'h3_resolution': 8,
    'horizontal_crs': 'H3/WGS84 (EPSG:4326 source raster)',
    'spatial_bounds_wgs84': tuple(INPUT_BOUNDS),
    'bathymetry_units': 'm', 'bathymetry_sign': 'positive_down',
    'minimum_depth_m': float(output['BATHYMETRY'].min()),
    'maximum_depth_m': float(output['BATHYMETRY'].max()),
    'mean_depth_m': float(output['BATHYMETRY'].mean()),
    'manifest_path': str(MANIFEST_PATH),
    'manifest_dataset_family': manifest['dataset_family'],
    'manifest_run_id': manifest['run_id'],
    'source_completeness': manifest['source_completeness'],
}, name='output')
display(output_summary.to_frame('value'))
display(schema)
display(output.head(8))
display(output.select_dtypes(include='number').describe().T)

## 10–11. Visualize and compare input with output

The panels make the transformation human-readable: a smooth negative-elevation raster becomes positive-down depth summaries on the declared H3 support.

In [ ]:
geometry = [
    Polygon([(longitude, latitude) for latitude, longitude in h3.cell_to_boundary(cell)])
    for cell in output['H3_INDEX'].astype(str)
]
mapped = gpd.GeoDataFrame(output[['H3_INDEX', 'BATHYMETRY']], geometry=geometry, crs='EPSG:4326')
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
raster_image = axes[0].imshow(
    input_values, extent=(INPUT_BOUNDS.left, INPUT_BOUNDS.right, INPUT_BOUNDS.bottom, INPUT_BOUNDS.top),
    cmap='Blues_r',
)
axes[0].set_title('Input: negative elevation (m)')
axes[0].set_xlabel('longitude'); axes[0].set_ylabel('latitude')
fig.colorbar(raster_image, ax=axes[0], shrink=0.8)
mapped.plot(column='BATHYMETRY', cmap='Blues', legend=True, edgecolor='white', linewidth=0.25, ax=axes[1])
axes[1].set_title('Output: H3 r8 positive-down depth (m)')
axes[1].set_xlabel('longitude'); axes[1].set_ylabel('latitude')
plt.show()

## 12. Validation checks

Every displayed PASS is backed by a boolean check. A failed required check raises after the table.

In [ ]:
fraction_columns = [column for column in output if column.startswith('BATHYMETRY_FRAC_')]
fraction_totals = output[fraction_columns].sum(axis=1, min_count=1)
valid_depth = output['BATHYMETRY'].dropna()
valid_fraction_totals = fraction_totals.dropna()
CHECKS.update({
    'Output exists': OUTPUT_PATH.is_file() and MANIFEST_PATH.is_file(),
    'Output schema valid': {'H3_INDEX', 'BATHYMETRY', 'BATHYMETRY_PIXEL_COUNT'}.issubset(output.columns) and output['H3_INDEX'].is_unique,
    'Spatial metadata valid': INPUT_CRS.to_epsg() == 4326 and output['H3_INDEX'].map(h3.get_resolution).eq(8).all(),
    'No impossible values': valid_depth.ge(0).all() and valid_depth.le(230.1).all() and np.allclose(valid_fraction_totals, 1.0),
})
validation_table = pd.DataFrame({
    'Check': list(CHECKS),
    'Result': ['PASS' if result else 'FAIL' for result in CHECKS.values()],
})
display(validation_table)
failed = [name for name, result in CHECKS.items() if not result]
if failed:
    raise AssertionError('Required validation checks failed: ' + ', '.join(failed))

## 13. Final summary

Warnings remain distinct from failures: this workflow deliberately does not acquire live GEBCO data, rebuild the regional case study, publish a canonical release, or validate downstream application integration.

In [ ]:
ELAPSED_SECONDS = time.perf_counter() - STARTED_AT
WARNINGS = [
    'Synthetic fixture only; no live provider availability was tested.',
    'This is not a regional scientific rebuild or canonical release audit.',
]
summary = pd.Series({
    'Toolkit': 'toolkit-seascape',
    'Version': PACKAGE_VERSION,
    'Validation mode': 'offline deterministic smoke/acceptance',
    'Workspace': str(WORKSPACE),
    'Representative pipeline': 'bathymetry.run_pipeline (local GeoTIFF → H3 Parquet)',
    'Input pixels': int(input_values.count()),
    'Output rows': len(output),
    'Validation checks passed': f'{sum(CHECKS.values())}/{len(CHECKS)}',
    'Warnings': ' | '.join(WARNINGS),
    'Execution seconds': round(ELAPSED_SECONDS, 2),
    'Execution status': 'PASS',
}, name='validation summary')
display(summary.to_frame('value'))
print('SEASCAPE TOOLKIT VALIDATION: PASS')